# HealthYarn Batch TTS Generator (YarnGPT2b on Google Colab GPU)

This notebook runs `backend/scripts/generate_audio.py` on a Colab T4/A100 GPU using `saheedniyi/YarnGPT2b` to synthesize approved Yorùbá, Igbo, and Nigerian English health answers into Opus `.ogg` clips (24–32 kbps mono, < 150 KB each).

**Hard rule**: YarnGPT is used offline for text-to-speech only, never at runtime and never for ASR.

In [ ]:
# 1. Mount Google Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

!apt-get -y install ffmpeg
!git clone https://github.com/saheedniyi02/yarngpt.git /content/yarngpt_repo
!pip install -q outetts uroman torchaudio transformers accelerate gdown sqlmodel pydantic-settings numpy

In [ ]:
# 2. Download WavTokenizer configuration & weights required by YarnGPT2b
!wget -q https://huggingface.co/novateur/WavTokenizer-medium-speech-75token/resolve/main/wavtokenizer_mediumdata_frame75_3s_nq1_code4096_dim512_kmeans200_attn.yaml -O /content/wavtokenizer_mediumdata_frame75_3s_nq1_code4096_dim512_kmeans200_attn.yaml
!gdown -q 1-ASeEkrn4HY49yZWHTASgfGFNXdVnLTt -O /content/wavtokenizer_large_speech_320_24k.ckpt

In [ ]:
# 3. Point to your HealthYarn repository folder in Drive (or clone your fork)
import os, sys
REPO_DIR = '/content/drive/MyDrive/HealthYarn'
sys.path.insert(0, '/content/yarngpt_repo')
os.chdir(f'{REPO_DIR}/backend')

# Seed database if needed, then generate YarnGPT Opus clips (2 takes per answer)
!python -m scripts.seed --demo
!python -m scripts.generate_audio --only-missing --takes 2

In [ ]:
# 4. Zip generated .ogg files for download back into backend/static/audio/
!cd /content/drive/MyDrive/HealthYarn/backend/static && zip -r /content/healthyarn_audio_ogg.zip audio/
from google.colab import files
files.download('/content/healthyarn_audio_ogg.zip')